In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm
import multiprocessing as mp
from transformers import get_linear_schedule_with_warmup

# %env KAGGLE_IS_COMPETITION_RERUN ='true'

In [ ]:
MAX_LEN = 256
BATCH_SIZE = 24
EPOCHS =  5
U_EPOCHS = 1
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
SEEDS = [42, 123]

In [ ]:
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [ ]:
df = pd.read_csv(train_path)
df['rule']= df['rule'].str.lower().str.strip()
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [ ]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [ ]:
test_df = pd.read_csv(test_path)
test_df['rule']= test_df.rule.str.lower().str.strip()

augmented_train = add_data(df)
augmented_test = add_data(test_df)

augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]

augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',
    'label': 'mean'
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])


augmented_df.head()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [ ]:
unlabelled= pd.read_csv('/kaggle/input/199kv2-jigsaw/test.csv').iloc[31750:]
# unlabelled2= pd.read_csv('/kaggle/input/jigsaw500k/test.csv')
# unlabelled2['row_id']+= len(unlabelled)

# unlabelled3= pd.read_csv('/kaggle/input/jigsaw500k2/test.csv')
# unlabelled3['row_id']+= unlabelled2.row_id.iloc[-1]+1


# unlabelled= pd.concat([unlabelled1],axis=0,ignore_index=True).iloc[:]#unlabelled3
# unlabelled= pd.concat([unlabelled1,unlabelled2,unlabelled3],axis=0,ignore_index=True).iloc[:]

unlabelled['rule']= unlabelled['rule'].str.lower().str.strip()
unlabelled['text']= unlabelled['rule']+ ' [SEP] '+ unlabelled['body']

rule_map= {i:j for j,i in enumerate(unlabelled.rule.unique())}
augmented_df['rule_id']= augmented_df.rule.map(rule_map)
unlabelled['rule_id']= unlabelled.rule.map(rule_map)

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    for seed in SEEDS:
        train_data, val_data = train_test_split(
            augmented_df, 
            test_size=0.2, 
            stratify=augmented_df["rule"], 
            random_state=seed
        )
        unlabelled_seed= unlabelled.copy()
        unlabelled_seed['label']= unlabelled_seed.rule_violation.round(0)
        # # change to keep the actual test set out of llm predictions.
        # unlabelled_seed= unlabelled_seed.query('text not in @augmented_df.text.values')

        # temp_test_data= test_df["rule"].str.lower().str.strip() + " [SEP] " + test_df["body"]
        # unlabelled_seed= unlabelled_seed.query('text not in @temp_test_data')

        # confident_positives = unlabelled_seed[unlabelled_seed['rule_violation'] >= 0.85]

        # # Take confident negatives (pred <= 0.1)
        # confident_negatives = unlabelled_seed[unlabelled_seed['rule_violation'] <= 0.15]
        
        # # Sample same number of negatives + 10%
        # n_positives = len(confident_positives)
        # n_negatives_to_sample = int(n_positives * 1.1)
        
        # sampled_negatives = confident_negatives.sample(
        #   n=min(n_negatives_to_sample, len(confident_negatives)),
        #   random_state=seed
        # )
        
        # # Combine
        # unlabelled_taken = pd.concat([confident_positives, sampled_negatives], ignore_index=True)
        unlabelled_taken= unlabelled_seed
        
        # unlabelled_taken= unlabelled_taken.sample(frac=.8, random_state=seed)# take 80% of all the points for data diversity
        # print(f'Seed {seed} - Pseudo: {len(confident_positives)} pos (>=0.85), {len(sampled_negatives)} neg (<=0.15), total={len(unlabelled_taken)}')

        train_data.to_csv(f'fixed_train_split_seed_{seed}.csv', index=False)
        val_data.to_csv(f'fixed_val_split_seed_{seed}.csv', index=False)
        unlabelled_taken.to_csv(f'fixed_pseudo_seed_{seed}.csv', index=False)
        print(f'Seed {seed} splits saved: train={len(train_data)}, val={len(val_data)}, pseudo={len(unlabelled_taken)}')

In [ ]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len,weights=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids
        self.weights = weights if weights is not None else [1.0]*len(texts)

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        item['weights'] = torch.tensor(self.weights[idx], dtype=torch.float)
        return item

In [ ]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, device, scaler):  # Pass scaler in
    model.train()
    total_loss = 0

    for batch in tqdm(loader, desc='Training'):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        weights = batch["weights"].to(device)

        with torch.cuda.amp.autocast():
            logits = model(input_ids, mask)
            loss = nn.BCEWithLogitsLoss(reduction='none')(logits, labels)
            loss = (loss * weights).mean()

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)  # ← FIX: Unscale before clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
def validate(model, loader, device):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            rule_ids = batch["rule_ids"]
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            rule_aucs[rule_id] = np.nan
    
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    return avg_auc_per_rule, val_loss, preds

In [ ]:
def train_model_seed(seed, gpu_id):
    import random
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    device = torch.device(f"cuda:{gpu_id}")
    print(f"[Seed {seed}] Training on {device}")
    
    train_data = pd.read_csv(f'fixed_train_split_seed_{seed}.csv')
    val_data = pd.read_csv(f'fixed_val_split_seed_{seed}.csv')
    unlabelled_taken = pd.read_csv(f'fixed_pseudo_seed_{seed}.csv')
    
    train_ds = JigsawDataset(
        train_data['text'].tolist(),#+unlabelled_taken['text'].tolist(), 
        train_data['label'].tolist(),#+unlabelled_taken['rule_violation'].tolist(), 
        train_data['rule_id'].tolist(),#+unlabelled_taken['rule_id'].tolist(), 
        tokenizer, MAX_LEN,
        [1.0]*len(train_data),# + [.5]*len(unlabelled_taken)

    )
    # alpha = compute_alpha_from_soft_labels(torch.tensor(train_data['label'].tolist()))

    val_ds = JigsawDataset(
        val_data['text'].tolist(), 
        val_data['label'].tolist(), 
        val_data['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    scaler = torch.cuda.amp.GradScaler()
    model = JigsawModel(MODEL_PATH).to(device)
    # Load the state dict
    state_dict = torch.load(
    f"/kaggle/input/deberta-base-923-pseudo/model_pseudo_deberta_base_seed33.bin",
    map_location=device
    )
    
    # Remove 'module.' prefix from all keys
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    
    # Load into your model
    model.load_state_dict(state_dict)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False

    
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    best_auc = 0
    best_loss= None
    for epoch in range(EPOCHS):
        print(f"[Seed {seed}] Epoch {epoch+1}/{EPOCHS}")
        loss = train_one_epoch(model, train_loader, optimizer, scheduler, device, scaler)
        val_auc, val_loss, val_preds = validate(model, val_loader, device)
        
        print(f"[Seed {seed}] Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            best_loss= val_loss
            torch.save(model.state_dict(), f"model_seed_{seed}.bin")
    
    print(f"[Seed {seed}] Best validation AUC: {best_auc:.4f}")
    import json
    with open(f'results_seed_{seed}.json', 'w') as f:
        json.dump({'seed': seed, 'best_auc': best_auc, 'best_loss': best_loss}, f)

    return seed, best_auc

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import torch.multiprocessing as mp
    mp.set_start_method('fork', force=True)
    
    # processes = []
    # for idx, seed in enumerate(SEEDS):
    #    gpu_id = idx % torch.cuda.device_count()
    #    p = mp.Process(target=train_model_seed_pseudo, args=(seed, gpu_id))
    #    p.start()
    #    processes.append(p)
    
    # for p in processes:
    #    p.join()

    processes = []
    for idx, seed in enumerate(SEEDS):
       gpu_id = idx % torch.cuda.device_count()
       p = mp.Process(target=train_model_seed, args=(seed, gpu_id))
       p.start()
       processes.append(p)
    
    for p in processes:
       p.join()
    # train_model_seed_pseudo(42,0)
    # for seed in SEEDS:
    #     train_model_seed(seed, 0)
    import json
    results = []
    for seed in SEEDS:
      with open(f'results_seed_{seed}.json', 'r') as f:
          results.append(json.load(f))
    
    aucs = [r['best_auc'] for r in results]
    losses = [r['best_loss'] for r in results]
    
    print(f"AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    print(f"Loss: {np.mean(losses):.4f} ± {np.std(losses):.4f}")

    print("All models trained!")

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df_test = pd.read_csv(test_path)
    df_test['rule']= df_test['rule'].str.lower().str.strip()
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test), [0]*len(df_test), tokenizer, MAX_LEN)
    test_loader = DataLoader(test_ds, batch_size=32)
    
    all_preds = []
    
    for seed in SEEDS:
        device = torch.device("cuda:0")
        model = JigsawModel(MODEL_PATH).to(device)
        model.load_state_dict(torch.load(f"model_seed_{seed}.bin", map_location=device))
        model.eval()
        
        test_preds = []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Inference seed {seed}"):
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                logits = model(ids, mask)
                test_preds.extend(torch.sigmoid(logits).cpu().numpy())
        
        all_preds.append(test_preds)
    
    ensemble_preds = np.mean(all_preds, axis=0)
    
    sample = pd.read_csv(sample_sub_path)
    sample["rule_violation"] = ensemble_preds
    sample.to_csv("submission.csv", index=False)
    print(f"Ensembled {len(SEEDS)} models")
else:
    !touch submission.csv
    
!head -n 4 submission.csv